# TradeSense · Component 03 · notebook 57: does history tell how Alert-prone the next hour is? (protocol H2 v1; CPU, minutes)

**Question.** Notebook 53 reported H-H5 as FAILED because the rule "at least 3 of 5 neighbours were followed by an Alert" was true for only 0.5% of hours. Protocol H2 v1 (written before any number) uses a graded score instead: the share of the 10 / 20 / 40 nearest past windows followed by an Alert, chosen on CAL (August) and evaluated once on Sep-Oct. The retrieval, the descriptor c2 and every frozen part are unchanged. The result of notebook 53 is not edited.

**Order of work.** (1) protocol hash, (2) genuine timeline and descriptors, (3) unit tests, (4) score chosen on CAL, (5) TEST-B evaluated once.

**Protocol text hashed by this notebook:**

```
# Protocol H2 v1 (go-ahead given by the user 2026-10-18; frozen when notebook 57 hashes this exact text) - does what followed similar past windows tell how Alert-prone the next hour is? (novelty 4, follow-up to hypothesis H-H5 of protocol H1 v4)
Written BEFORE any number of this analysis exists. Nothing frozen changes (models, 6 features, normalisation, fusion stacker, thresholds q, Layer 3 sets) and the retrieval itself is the validated one of H1 v4 / notebook 53 (descriptor c2, gap 120, past-only). CPU, no training. Protocol H1 v4 and notebook 53 (H-H5 FAILED: only 0.5% of hours had >= 3 of 5 neighbours followed by an Alert, lift 1.97 with interval 0.00..6.19) stay as reported and are not edited. Thresholds below are not lowered to make the test easier.

## Why H-H5 failed (diagnosis from notebook 53)
The rule "at least 3 of the 5 nearest windows were followed by an Alert" is almost never true (0.5% of hours), because an Alert is a rare event (the Alert line is a high quantile of the genuine scores). A rule that is true for so few hours cannot reach a lower bound above 1. The information is probably present in a graded form: the fraction of MANY neighbours followed by an Alert (not a yes/no on 5) and a ranking of hours instead of one cut.

## Part A - the score (declared now; chosen on CAL genuine hourly queries only)
- Queries: genuine hourly windows (end bar on every full hour) as in notebook 53: CAL = August (from the 1,440th bar), TEST-B = Sep and Oct (from the 1,440th bar of Sep); archive = genuine past windows only; the neighbours are the validated top-k with gap 120 bars.
- Outcome y = 1 if the query hour is followed by at least one minute above the declared Alert line within the next 60 bars (forward_outcomes of notebook 53). Persistence reference = rank of the maximum fused score in the query window.
- Score p_hat(k, weighting) over the k nearest neighbours: share of neighbours whose next hour contained an Alert minute. Weighting "u": uniform; weighting "d": weights exp(-d_i / mean(d_1..d_k)). k in {10, 20, 40}. Six candidates.
- Selection (CAL only, TEST not loaded for this analysis): highest mean AUC of p_hat for y over BTC and ETH on CAL queries; ties go to the smaller k and then to "u". A CAL query set must contain at least 30 positive hours per symbol, otherwise the notebook stops and reports that the analysis cannot be decided. The pooled CAL 90th percentile of p_hat per symbol (thr90) is stored with the choice before TEST is evaluated.
- Combination for H-H5c: logistic regression (no penalty beyond a ridge of 1e-6) of y on [rank(persistence)] versus on [rank(persistence), p_hat], fitted on CAL queries per symbol and applied unchanged to TEST-B.

## Part B - hypotheses on TEST-B hourly queries (new analysis on saved scores)
- H-H5b (the score ranks hours): AUC of p_hat for y >= 0.60, mean of BTC and ETH, day-block 95% lower bound > 0.50 (2,000 draws, seed 0, units = days within symbol).
- H-H5c (adds to persistence): AUC of the two-variable logistic model minus AUC of the persistence-only model >= 0.02, day-block 95% lower bound > 0.
- H-H5d (top group of hours): hours with p_hat >= thr90 and p_hat > 0 versus all other hours: lift in the actual rate of an Alert in the next hour >= 2.0, day-block 95% lower bound > 1 (mean of BTC and ETH).
- Decidability: if TEST-B has fewer than 50 positive hours per symbol, H-H5b..d are reported as "cannot be decided" and counted as not holding.
- Descriptive (no decision): calibration table of p_hat in five bins (observed Alert rate per bin), Brier score and Brier skill against the base rate, share of hours flagged, the persistence-rule lift, number of positive hours, results per symbol.
- Stated limits: only two symbols and a short history; the Alert outcome on genuine data is a false-alarm tendency, not manipulation; neighbours that are close in time are partly the same episode (gap 120 only); drift between July and October.

## Product rule (declared now)
If H-H5b and H-H5d hold, the history tab may add one line, "In X of the N most similar past windows an Alert followed within the hour" (counts, never a probability of manipulation, never a forecast). If they fail, that line is not shown and the tab keeps showing only the validated context.

```


In [1]:
import os, sys, json, time, hashlib, warnings, glob
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
PROTO_H2 = r"""# Protocol H2 v1 (go-ahead given by the user 2026-10-18; frozen when notebook 57 hashes this exact text) - does what followed similar past windows tell how Alert-prone the next hour is? (novelty 4, follow-up to hypothesis H-H5 of protocol H1 v4)
Written BEFORE any number of this analysis exists. Nothing frozen changes (models, 6 features, normalisation, fusion stacker, thresholds q, Layer 3 sets) and the retrieval itself is the validated one of H1 v4 / notebook 53 (descriptor c2, gap 120, past-only). CPU, no training. Protocol H1 v4 and notebook 53 (H-H5 FAILED: only 0.5% of hours had >= 3 of 5 neighbours followed by an Alert, lift 1.97 with interval 0.00..6.19) stay as reported and are not edited. Thresholds below are not lowered to make the test easier.

## Why H-H5 failed (diagnosis from notebook 53)
The rule "at least 3 of the 5 nearest windows were followed by an Alert" is almost never true (0.5% of hours), because an Alert is a rare event (the Alert line is a high quantile of the genuine scores). A rule that is true for so few hours cannot reach a lower bound above 1. The information is probably present in a graded form: the fraction of MANY neighbours followed by an Alert (not a yes/no on 5) and a ranking of hours instead of one cut.

## Part A - the score (declared now; chosen on CAL genuine hourly queries only)
- Queries: genuine hourly windows (end bar on every full hour) as in notebook 53: CAL = August (from the 1,440th bar), TEST-B = Sep and Oct (from the 1,440th bar of Sep); archive = genuine past windows only; the neighbours are the validated top-k with gap 120 bars.
- Outcome y = 1 if the query hour is followed by at least one minute above the declared Alert line within the next 60 bars (forward_outcomes of notebook 53). Persistence reference = rank of the maximum fused score in the query window.
- Score p_hat(k, weighting) over the k nearest neighbours: share of neighbours whose next hour contained an Alert minute. Weighting "u": uniform; weighting "d": weights exp(-d_i / mean(d_1..d_k)). k in {10, 20, 40}. Six candidates.
- Selection (CAL only, TEST not loaded for this analysis): highest mean AUC of p_hat for y over BTC and ETH on CAL queries; ties go to the smaller k and then to "u". A CAL query set must contain at least 30 positive hours per symbol, otherwise the notebook stops and reports that the analysis cannot be decided. The pooled CAL 90th percentile of p_hat per symbol (thr90) is stored with the choice before TEST is evaluated.
- Combination for H-H5c: logistic regression (no penalty beyond a ridge of 1e-6) of y on [rank(persistence)] versus on [rank(persistence), p_hat], fitted on CAL queries per symbol and applied unchanged to TEST-B.

## Part B - hypotheses on TEST-B hourly queries (new analysis on saved scores)
- H-H5b (the score ranks hours): AUC of p_hat for y >= 0.60, mean of BTC and ETH, day-block 95% lower bound > 0.50 (2,000 draws, seed 0, units = days within symbol).
- H-H5c (adds to persistence): AUC of the two-variable logistic model minus AUC of the persistence-only model >= 0.02, day-block 95% lower bound > 0.
- H-H5d (top group of hours): hours with p_hat >= thr90 and p_hat > 0 versus all other hours: lift in the actual rate of an Alert in the next hour >= 2.0, day-block 95% lower bound > 1 (mean of BTC and ETH).
- Decidability: if TEST-B has fewer than 50 positive hours per symbol, H-H5b..d are reported as "cannot be decided" and counted as not holding.
- Descriptive (no decision): calibration table of p_hat in five bins (observed Alert rate per bin), Brier score and Brier skill against the base rate, share of hours flagged, the persistence-rule lift, number of positive hours, results per symbol.
- Stated limits: only two symbols and a short history; the Alert outcome on genuine data is a false-alarm tendency, not manipulation; neighbours that are close in time are partly the same episode (gap 120 only); drift between July and October.

## Product rule (declared now)
If H-H5b and H-H5d hold, the history tab may add one line, "In X of the N most similar past windows an Alert followed within the hour" (counts, never a probability of manipulation, never a forecast). If they fail, that line is not shown and the tab keeps showing only the validated context.
"""
PHH = hashlib.sha256(PROTO_H2.encode()).hexdigest(); pp = OUT/'protocol_h2_v1.md'
if pp.exists(): assert pp.read_text() == PROTO_H2, 'protocol H2 v1 exists and differs from this notebook'
else: pp.write_text(PROTO_H2)
(OUT/'protocol_h2_v1.sha256').write_text(PHH + '  protocol_h2_v1.md\n'); assert (OUT/'h1v4_results.json').exists(), 'run notebook 53 first'
for f in ['test_b_results_v1.json', 'declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'b2_lib.py', 'c03_infer.py']: assert (OUT/f).exists(), f'{f} missing'
T32 = json.loads((OUT/'test_b_results_v1.json').read_text())
for s in SYMBOLS: assert sha_file(OUT/'scores'/f'test_b_{s}.npz') == T32['test_score_files_sha16'][s], 'saved TEST scores differ from what notebook 32 recorded'
assert sha_file(OUT/'c03_infer.py') == T32['library_sha16'] and sha_file(OUT/'b2_lib.py') == T32['b2_lib_sha16'], 'library files differ from the ones that opened TEST'
sys.path.insert(0, str(OUT)); import b2_lib as B; import c03_infer as c
say('protocol H2 v1 sha256 ' + PHH[:16] + '; protocol B2 intact ' + ph[:16] + '; library ' + T32['library_sha16'] + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol H2 v1 sha256 4c60debad6862e3e; protocol B2 intact 443f99d5faf5929f; library 3b5d9df23521adf9


## 1 · Definitions (from notebook 53)

In [2]:
W = 60; OUTW = 60; GAP = 120; K = 5; WARM = B.WARM; AOLEN = 31 * 1440; AOFULL = 132480; TEST_LO = AOLEN; DEV_LO, DEV_HI = 90720, 129600; DEVN = DEV_HI - DEV_LO; HIST_MIN = 1500
SERIES = ['fused', 'AE', 'XGBoost', 'Transformer']; DNAMES = [f'{s}_{st}' for s in SERIES for st in ('mean', 'max', 'last')] + [f'maxabs_z_{f}' for f in B.FEATS]; assert len(DNAMES) == 18
SPEC = {'window_bars': W, 'outcome_bars': OUTW, 'gap_bars': GAP, 'k': K, 'dims': DNAMES, 'metric': 'euclidean on per-symbol standardised descriptor', 'history_min_bars': HIST_MIN}
def raw_desc(F4, zabs, hist_min=HIST_MIN):
    """(n, 18) window descriptors for windows ending at each bar (NaN where invalid). F4 = 4 score arrays; zabs = (n, 6) |z|."""
    n = len(F4[0]); cols = []
    for x in F4:
        s = pd.Series(np.asarray(x, 'float64')); r = s.rolling(W, min_periods=W); cols += [r.mean().to_numpy(), r.max().to_numpy(), s.to_numpy()]
    zm = pd.DataFrame(np.asarray(zabs, 'float64')).rolling(W, min_periods=W).max().to_numpy(); D_ = np.column_stack(cols + [zm[:, j] for j in range(6)])
    D_[np.arange(n) < hist_min + W - 1] = np.nan; return D_
def forward_outcomes(F, thrA, logret):
    """for a window ending at e: next OUTW bars e+1..e+OUTW -> peak fused, minutes above Alert line, Alert reached, realised vol. NaN if the bars do not all exist."""
    n = len(F); pk = np.full(n, np.nan); ma = np.full(n, np.nan); vo = np.full(n, np.nan)
    if n > OUTW:
        from numpy.lib.stride_tricks import sliding_window_view as swv
        Fw = swv(F, OUTW)[1:]; Aw = swv((F > thrA).astype(float), OUTW)[1:]; Lw = swv(logret, OUTW)[1:]; m = len(Fw)
        pk[:m] = Fw.max(axis=1); ma[:m] = Aw.sum(axis=1); vo[:m] = Lw.std(axis=1)
    return pk, ma, vo
def select(dist, times, k=K, gap=GAP):
    """indices of up to k nearest candidates, nearest first, every pair >= gap bars apart"""
    order = np.argsort(dist, kind='stable'); picked = []
    for i in order:
        if all(abs(times[i] - times[j]) >= gap for j in picked): picked.append(i)
        if len(picked) == k: break
    return picked
def retrieve(A, At, q, q_end, k=K, gap=GAP):
    """A: standardised archive (n, d) sorted by end time At; q: query descriptor; past-only: archive end + OUTW < query start (q_end - W + 1)"""
    n = int(np.searchsorted(At, q_end - W + 1 - OUTW, side='left'))      # At < q_start - OUTW
    if n == 0: return np.array([], int), np.array([])
    d = np.sqrt(((A[:n] - q) ** 2).sum(axis=1)); m = min(n, 600); part = np.argpartition(d, m - 1)[:m] if n > m else np.arange(n)
    sel = select(d[part], At[part], k, gap)
    if len(sel) < k and n > m: sel = select(d, At[:n], k, gap); return np.array(sel, int), d[sel]
    ii = part[sel]; return np.array(ii, int), d[ii]

## 2 · Genuine timeline

In [3]:
DF = json.load(open(OUT/'declared_final_set_v1.json')); FUSD = json.load(open(OUT/'declared_fusion_v0.json')); DW = json.load(open(OUT/'declared_watch_v1.json'))
MODELS = c.MODEL_NAMES; FNORM = {m: DF['models'][m]['normalisation'] for m in MODELS}; FILES = {'AE-ctx fmax': 'ae_ctx_agg', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}; TAGM = dict(zip(MODELS, ['AE', 'XGB', 'TR']))
QA = FUSD['q_T1_by_budget']['2']; QW = DW['watch']['q']['4']
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:WARM] = np.nan; return a
def find1(pat):
    cc = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert cc, pat; return sorted(cc)[0]
def fix(d): d = d.copy(); d.loc[d.index[:15], 'volume_acceleration'] = 0.0; return d
def mcol(zz, k): x = zz[k]; return x[:, 0] if x.ndim == 2 else x
G = {}      # per symbol: genuine timeline (DEV then Aug-Oct) and injected series for the H-H1 evaluation
for s in SYMBOLS:
    a = fix(pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet'))); a = a.set_index('datetime') if 'datetime' in a.columns else a
    b = fix(pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet')); assert len(b) == AOFULL, len(b)
    N0 = B.fit_n0(a[B.FEATS].reset_index(drop=True).iloc[15:90720])
    zL1 = np.abs(B.normalise(a[B.FEATS].reset_index(drop=True), 'N1', N0)); zAO = np.abs(B.normalise(b[B.FEATS].reset_index(drop=True), 'N1', N0))
    zf = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); tz = np.load(OUT/'scores'/f'test_b_{s}.npz'); assert 'FUS__GEN' in tz.files
    dev = [nanw(zf['OOF_GEN_L1'])[DEV_LO:DEV_HI]]; ao = [nanw(tz['FUS__GEN'])]
    for m in MODELS:
        zz = np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz'); dev.append(nanw(mcol(zz, 'GEN_L1'))[DEV_LO:DEV_HI]); ao.append(nanw(tz[TAGM[m] + '__GEN']))
    # parity: the August part of the saved TEST file equals the fusion August scores of notebook 35
    dmax = np.nanmax(np.abs(ao[0][:AOLEN] - nanw(zf['GEN_AO'])[:AOLEN])); say(f'{s}: parity of August fused scores (TEST file vs notebook 35): max difference {dmax:.2e}')
    ts = np.concatenate([a.index.to_numpy()[DEV_LO:DEV_HI], b.index.to_numpy()]).astype('datetime64[ns]')
    Fg = np.concatenate([dev[0], ao[0]]); thrA = c.rolling_threshold(Fg, QA[s]); thrW = c.rolling_threshold(Fg, QW[s])
    seg = np.concatenate([np.zeros(DEVN, int), np.ones(AOFULL, int)]); lr = np.concatenate([a['log_returns'].to_numpy()[DEV_LO:DEV_HI], b['log_returns'].to_numpy()])
    RD = np.vstack([raw_desc([x for x in [dev[0]] + dev[1:]], zL1[DEV_LO:DEV_HI], hist_min=0), raw_desc(ao[0:1] + ao[1:], zAO, hist_min=HIST_MIN)])
    pk, ma, vo = forward_outcomes(Fg[:DEVN], thrA[:DEVN], lr[:DEVN]); pk2, ma2, vo2 = forward_outcomes(Fg[DEVN:], thrA[DEVN:], lr[DEVN:]); PK, MA, VO = np.concatenate([pk, pk2]), np.concatenate([ma, ma2]), np.concatenate([vo, vo2])
    st = np.where(Fg > thrA, 2, np.where(Fg > thrW, 1, 0)); stp = pd.Series(st).rolling(W, min_periods=1).max().to_numpy().astype(int)
    G[s] = dict(ts=ts, F=Fg, thrA=thrA, thrW=thrW, seg=seg, lr=lr, RD=RD, PK=PK, MA=MA, VO=VO, st=st, stp=stp, N0=N0, parts=(dev, ao), zL1=zL1, zAO=zAO, a=a, b=b)
    say(f'{s}: genuine timeline {len(Fg)} bars (DEV {DEVN} + Aug-Oct {AOFULL}); valid windows with outcome {int((np.isfinite(RD).all(axis=1) & np.isfinite(PK)).sum())}')

BTCUSDT: parity of August fused scores (TEST file vs notebook 35): max difference 9.61e-03
BTCUSDT: genuine timeline 171360 bars (DEV 38880 + Aug-Oct 132480); valid windows with outcome 169622
ETHUSDT: parity of August fused scores (TEST file vs notebook 35): max difference 7.44e-03
ETHUSDT: genuine timeline 171360 bars (DEV 38880 + Aug-Oct 132480); valid windows with outcome 169622


## 3 · Descriptors

In [4]:
# ---------------- extended descriptors (blocks S, ZA, ZM, ZL, SH) on the genuine timeline
from numpy.lib.stride_tricks import sliding_window_view as swv
BL = {'S': np.arange(0, 12), 'ZA': np.arange(12, 18), 'ZM': np.arange(18, 24), 'ZL': np.arange(24, 30), 'SH': np.arange(30, 32)}
def ext_desc(F4, zs, hist_min=HIST_MIN):
    """(n, 32): 12 score summaries | 6 max|z| | 6 mean signed z | 6 last signed z | 2 shape numbers; NaN where invalid"""
    zs = np.asarray(zs, 'float64'); n = len(F4[0]); base = raw_desc(F4, np.abs(zs), hist_min); ok = np.isfinite(base).all(axis=1)
    zm = pd.DataFrame(zs).rolling(W, min_periods=W).mean().to_numpy(); f = np.asarray(F4[0], 'float64'); pos = np.full(n, np.nan)
    if n >= W: pos[W - 1:] = np.argmax(swv(np.nan_to_num(f, nan=-1e9), W), axis=1) / (W - 1)
    m10 = pd.Series(f).rolling(10, min_periods=10).mean().to_numpy(); slope = m10 - pd.Series(m10).shift(W - 10).to_numpy()
    D = np.column_stack([base, zm, zs, pos, slope]); D[~ok] = np.nan; return D
CANDS = {}
for nm, blocks in [('c1', ['S', 'ZA']), ('c2', ['S', 'ZM', 'ZL']), ('c3', ['S', 'ZA', 'ZM']), ('c4', ['S', 'ZA', 'ZM', 'SH'])]:
    CANDS[nm] = (blocks, 1.0); CANDS[nm + 'w'] = (blocks, 0.5)
CANDS['b1 score-only'] = (['S'], 1.0)
def cspec(c):
    blocks, w = CANDS[c]; cols = np.concatenate([BL[b] for b in blocks]); wt = np.concatenate([np.full(len(BL[b]), 1.0 if b == 'S' else w) for b in blocks]); return cols, wt
EXS = {}; STDE = {}; ARC = {}
for s in SYMBOLS:
    g = G[s]; dev, ao = g['parts']; zL = np.asarray(B.normalise(g['a'][B.FEATS].reset_index(drop=True), 'N1', g['N0']), 'float64'); zA = np.asarray(B.normalise(g['b'][B.FEATS].reset_index(drop=True), 'N1', g['N0']), 'float64')
    EX = np.vstack([ext_desc(dev, zL[DEV_LO:DEV_HI], 0), ext_desc(ao, zA, HIST_MIN)])
    valid = np.isfinite(EX).all(axis=1) & np.isfinite(g['PK']) & np.isfinite(g['MA']) & np.isfinite(g['VO']); devm = valid & (np.arange(len(valid)) < DEVN)
    mu = EX[devm].mean(axis=0); sd = EX[devm].std(axis=0) + 1e-9; STDE[s] = (mu, sd); Z = (EX - mu) / sd; idx = np.flatnonzero(valid); EXS[s] = Z
    ARC[s] = dict(Z=Z[idx], t=idx.astype(np.int64), PK=g['PK'][idx], MA=g['MA'][idx], lab=np.zeros(len(idx), int))
    say(f'{s}: extended descriptors (32 numbers); standardiser from {int(devm.sum())} genuine DEV windows; genuine archive {len(idx)} windows')

BTCUSDT: extended descriptors (32 numbers); standardiser from 38761 genuine DEV windows; genuine archive 169622 windows
ETHUSDT: extended descriptors (32 numbers); standardiser from 38761 genuine DEV windows; genuine archive 169622 windows


## 4 · Unit tests

In [5]:
rg = np.random.default_rng(0); n = 3000; F4 = [rg.normal(size=n).cumsum() * 0.01 + i for i in range(4)]; zs_ = rg.normal(size=(n, 6)); Dd = ext_desc(F4, zs_, 0); e = 2000
assert np.allclose(Dd[e, 18:24], zs_[e - W + 1:e + 1].mean(axis=0)) and np.allclose(Dd[e, 24:30], zs_[e]) and abs(Dd[e, 30] - np.argmax(F4[0][e - W + 1:e + 1]) / (W - 1)) < 1e-12 and abs(Dd[e, 31] - (F4[0][e - 9:e + 1].mean() - F4[0][e - W + 1:e - W + 11].mean())) < 1e-12
assert np.allclose(Dd[e, :18], raw_desc(F4, np.abs(zs_), 0)[e]); say('PASS T01 extended descriptor (mean / last signed z, shape numbers) equals a hand calculation and contains the v3 descriptor')
F4b = [np.where(np.arange(n) > e, x * 5 + 3, x) for x in F4]; zb = np.where(np.arange(n)[:, None] > e, zs_ * 4, zs_); Db = ext_desc(F4b, zb, 0); assert np.allclose(Dd[:e + 1], Db[:e + 1], equal_nan=True); say('PASS T02 the descriptor at bar t does not depend on later bars')
cols, wt = cspec('c1'); assert list(cols) == list(range(18)) and (wt == 1).all(); cols, wt = cspec('c4w'); assert len(cols) == 26 and wt[:12].min() == 1 and wt[12:].max() == 0.5; assert cspec('b1 score-only')[0].tolist() == list(range(12)); say('PASS T03 candidate c1 is exactly the v3 descriptor; weights apply to feature blocks only')
for s in SYMBOLS:
    t_ = ARC[s]['t']; assert not ((t_ >= DEVN - OUTW) & (t_ < DEVN + HIST_MIN + W - 1)).any() and np.isfinite(ARC[s]['Z']).all()
    dm = np.isfinite(G[s]['RD'][:DEVN]).all(axis=1) & np.isfinite(G[s]['PK'][:DEVN]); assert STDE[s][0].shape == (32,)
say('PASS T04 no archive window spans the seam or lacks history; standardiser from genuine DEV windows only')
def _ref(ZA_, At, q, q_end):
    cand = np.flatnonzero(At + OUTW < q_end - W + 1); dist = np.sqrt(((ZA_[cand] - q) ** 2).sum(axis=1)); ref = []
    for j in np.argsort(dist, kind='stable'):
        if all(abs(At[cand[j]] - At[r]) >= GAP for r in ref): ref.append(cand[j])
        if len(ref) == 20: break
    return ref
At = np.unique(np.sort(rg.integers(0, 50000, 4000))); A_ = rg.normal(size=(len(At), 32)); q = rg.normal(size=32); ii, dd = retrieve(A_, At, q, 33333, k=20); assert list(ii) == [int(x) for x in _ref(A_, At, q, 33333)] and list(retrieve(A_, At, q, 33333, k=5)[0]) == list(ii[:5])
say('PASS T05 top-20 retrieval equals brute force and its first 5 equal the top-5 retrieval')

PASS T01 extended descriptor (mean / last signed z, shape numbers) equals a hand calculation and contains the v3 descriptor
PASS T02 the descriptor at bar t does not depend on later bars
PASS T03 candidate c1 is exactly the v3 descriptor; weights apply to feature blocks only
PASS T04 no archive window spans the seam or lacks history; standardiser from genuine DEV windows only
PASS T05 top-20 retrieval equals brute force and its first 5 equal the top-5 retrieval


## 5 · Results (protocol H2 v1)

In [6]:

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
H1 = json.loads((OUT/'h1v4_results.json').read_text()); assert H1['chosen'] == 'c2', H1['chosen']; CHOSEN = 'c2'; cols, wt = cspec(CHOSEN)
KS = [10, 20, 40]; WS = ['u', 'd']
def queries(s, lo, hi):
    g = G[s]; A = ARC[s]; Zw = A['Z'][:, cols] * wt; qe = np.arange(lo, hi); qe = qe[(qe - DEVN) % 60 == 0]
    qe = qe[np.isfinite(EXS[s][qe]).all(axis=1) & np.isfinite(g['PK'][qe]) & np.isfinite(g['MA'][qe])]
    P = {(k, w): np.full(len(qe), np.nan) for k in KS for w in WS}
    for j, e in enumerate(qe):
        q = EXS[s][e][cols] * wt; ii, dd = retrieve(Zw, A['t'], q, e, k=max(KS))
        for k in KS:
            if len(ii) < k: continue
            a_ = (A['MA'][ii[:k]] > 0).astype(float); P[(k, 'u')][j] = a_.mean(); wg = np.exp(-dd[:k] / max(dd[:k].mean(), 1e-12)); P[(k, 'd')][j] = float((wg * a_).sum() / wg.sum())
    return dict(q=qe, y=(g['MA'][qe] > 0).astype(float), pers=g['RD'][qe, 1], P=P, day=(qe - DEVN - TEST_LO) // 1440)
CALQ = {s: queries(s, DEVN + 1440, DEVN + AOLEN - OUTW - 1) for s in SYMBOLS}
posc = {s: int(CALQ[s]['y'].sum()) for s in SYMBOLS}; say('CAL genuine hourly queries per symbol ' + str({s: len(CALQ[s]['q']) for s in SYMBOLS}) + '; hours followed by an Alert ' + str(posc))
assert min(posc.values()) >= 30, 'fewer than 30 positive CAL hours per symbol: the analysis cannot be decided (protocol H2 v1)'
def auc(y, p):
    ok = np.isfinite(p) & np.isfinite(y)
    return roc_auc_score(y[ok], p[ok]) if ok.sum() > 10 and 0 < y[ok].sum() < ok.sum() else np.nan
SELR = []
for k in KS:
    for w in WS: SELR.append({'k': k, 'weighting': w, 'CAL AUC (mean of BTC, ETH)': float(np.mean([auc(CALQ[s]['y'], CALQ[s]['P'][(k, w)]) for s in SYMBOLS]))})
SELT = pd.DataFrame(SELR); best = SELT['CAL AUC (mean of BTC, ETH)'].max(); tied = SELT[SELT['CAL AUC (mean of BTC, ETH)'] >= best - 1e-12].sort_values(['k', 'weighting'], key=lambda x: x if x.name == 'k' else x.map({'u': 0, 'd': 1}), kind='stable')
KC, WC = int(tied.iloc[0]['k']), tied.iloc[0]['weighting']; say('--- score selection on CAL genuine hourly queries (TEST not evaluated yet) ---'); say(SELT.round(4).to_string(index=False)); say(f'chosen: k = {KC}, weighting = {WC}')
THR90 = {s: float(np.quantile(CALQ[s]['P'][(KC, WC)][np.isfinite(CALQ[s]['P'][(KC, WC)])], 0.9)) for s in SYMBOLS}
def upers(s, x): ref = np.sort(CALQ[s]['pers'][np.isfinite(CALQ[s]['pers'])]); return np.searchsorted(ref, x, side='right') / len(ref)
LR1, LR2 = {}, {}
for s in SYMBOLS:
    Q = CALQ[s]; p = Q['P'][(KC, WC)]; ok = np.isfinite(p) & np.isfinite(Q['pers'])
    LR1[s] = LogisticRegression(C=1e6, max_iter=1000).fit(upers(s, Q['pers'][ok])[:, None], Q['y'][ok]); LR2[s] = LogisticRegression(C=1e6, max_iter=1000).fit(np.column_stack([upers(s, Q['pers'][ok]), p[ok]]), Q['y'][ok])
DECL = {'status': 'DECLARED before TEST-B hourly queries are evaluated', 'protocol_h2_v1_sha256': PHH, 'k': KC, 'weighting': WC, 'thr90_cal': THR90, 'selection_table': SELT.to_dict('records'), 'descriptor': CHOSEN, 'index_versions': H1['index_versions']}
dp = OUT/'declared_history_alert_v1.json'
if dp.exists(): assert json.loads(dp.read_text())['k'] == KC and json.loads(dp.read_text())['weighting'] == WC, 'declared choice exists and differs'
else: dp.write_text(json.dumps(DECL, indent=1))
TESTQ = {s: queries(s, DEVN + TEST_LO + 1440, DEVN + AOFULL - OUTW - 1) for s in SYMBOLS}
say('TEST-B genuine hourly queries per symbol ' + str({s: len(TESTQ[s]['q']) for s in SYMBOLS}) + '; hours followed by an Alert ' + str({s: int(TESTQ[s]['y'].sum()) for s in SYMBOLS}))
DECIDABLE = all(TESTQ[s]['y'].sum() >= 50 for s in SYMBOLS)
def valid(s): Q = TESTQ[s]; return np.isfinite(Q['P'][(KC, WC)]) & np.isfinite(Q['pers'])
def stat_auc(s, ix): Q = TESTQ[s]; return auc(Q['y'][ix], Q['P'][(KC, WC)][ix])
def stat_gain(s, ix):
    Q = TESTQ[s]; ok = np.zeros(len(Q['q']), bool); ok[ix] = True; ok &= valid(s); p = Q['P'][(KC, WC)]
    return auc(Q['y'][ok], LR2[s].predict_proba(np.column_stack([upers(s, Q['pers'][ok]), p[ok]]))[:, 1]) - auc(Q['y'][ok], LR1[s].predict_proba(upers(s, Q['pers'][ok])[:, None])[:, 1])
def stat_lift(s, ix):
    Q = TESTQ[s]; p = Q['P'][(KC, WC)][ix]; y = Q['y'][ix]; ok = np.isfinite(p); fl = (p >= THR90[s]) & (p > 0) & ok
    return y[fl].mean() / y[~fl & ok].mean() if fl.any() and (~fl & ok).any() and y[~fl & ok].mean() > 0 else np.nan
def agg(fn, ixs): return float(np.nanmean([fn(s, ixs[s]) for s in SYMBOLS]))
ALL = {s: np.arange(len(TESTQ[s]['q'])) for s in SYMBOLS}
def dboot(fn, n=NBOOT, seed=0):
    rg = np.random.default_rng(seed); ud = {s: np.unique(TESTQ[s]['day']) for s in SYMBOLS}; out = []
    for _ in range(n):
        ixs = {}
        for s in SYMBOLS:
            pick = rg.choice(ud[s], len(ud[s])); ixs[s] = np.concatenate([np.flatnonzero(TESTQ[s]['day'] == d) for d in pick])
        out.append(agg(fn, ixs))
    return np.nanpercentile(out, [2.5, 97.5])
A_, G_, L_ = agg(stat_auc, ALL), agg(stat_gain, ALL), agg(stat_lift, ALL); cA, cG, cL = dboot(stat_auc), dboot(stat_gain), dboot(stat_lift)
HB = bool(DECIDABLE and A_ >= 0.60 and cA[0] > 0.50); HC = bool(DECIDABLE and G_ >= 0.02 and cG[0] > 0); HD = bool(DECIDABLE and np.isfinite(L_) and L_ >= 2.0 and cL[0] > 1)
# ---------- descriptive
per = []
for s in SYMBOLS:
    Q = TESTQ[s]; ok = valid(s); p = Q['P'][(KC, WC)]; per.append({'symbol': s, 'queries': int(ok.sum()), 'positive hours': int(Q['y'][ok].sum()), 'AUC of p_hat': stat_auc(s, ALL[s]), 'AUC persistence only': auc(Q['y'][ok], upers(s, Q['pers'][ok])), 'gain from adding p_hat': stat_gain(s, ALL[s]), 'lift top group': stat_lift(s, ALL[s]), 'share flagged': float(((p >= THR90[s]) & (p > 0))[ok].mean())})
PERT = pd.DataFrame(per); PERT.to_csv(OUT/'h2_per_symbol.csv', index=False); say(''); say('--- TEST-B genuine hourly queries ---'); say(PERT.round(3).to_string(index=False))
pp_ = np.concatenate([TESTQ[s]['P'][(KC, WC)][valid(s)] for s in SYMBOLS]); yy = np.concatenate([TESTQ[s]['y'][valid(s)] for s in SYMBOLS])
try: bins = pd.qcut(pp_, 5, duplicates='drop'); CAL_ = pd.DataFrame({'bin': bins, 'y': yy, 'p': pp_}).groupby('bin', observed=True).agg(hours=('y', 'size'), mean_p_hat=('p', 'mean'), observed_alert_rate=('y', 'mean')).reset_index(); say(''); say('calibration of p_hat on TEST-B (descriptive):'); say(CAL_.round(3).to_string(index=False))
except Exception as ex: say('calibration table not available: ' + str(ex)); CAL_ = pd.DataFrame()
base = np.mean([CALQ[s]['y'].mean() for s in SYMBOLS]); br = np.mean([np.mean((LR2[s].predict_proba(np.column_stack([upers(s, TESTQ[s]['pers'][valid(s)]), TESTQ[s]['P'][(KC, WC)][valid(s)]]))[:, 1] - TESTQ[s]['y'][valid(s)]) ** 2) for s in SYMBOLS]); b0 = np.mean([np.mean((CALQ[s]['y'].mean() - TESTQ[s]['y'][valid(s)]) ** 2) for s in SYMBOLS])
say(f'(descriptive) Brier of the two-variable model {br:.4f} versus the CAL base rate {b0:.4f}: skill {1 - br / b0:+.3f}; CAL base rate of an Alert next hour {base:.3f}')
say(''); say('=== pre-registered hypotheses (protocol H2 v1; new analysis on saved TEST-B scores; H-H5 of H1 v4 stays reported as failed) ===')
if not DECIDABLE: say('TEST-B has fewer than 50 positive hours for a symbol: H-H5b..d cannot be decided and count as not holding.')
say(f'H-H5b AUC of p_hat >= 0.60 with day-block lower bound > 0.50: {A_:.3f} ({cA[0]:.3f}..{cA[1]:.3f}) -> {"HOLDS" if HB else "FAILS"}')
say(f'H-H5c AUC gain over persistence-only >= 0.02 with lower bound > 0: {G_:+.3f} ({cG[0]:+.3f}..{cG[1]:+.3f}) -> {"HOLDS" if HC else "FAILS"}')
say(f'H-H5d lift of an Alert next hour for hours with p_hat >= CAL 90th percentile >= 2.0 with lower bound > 1: {L_:.2f} ({cL[0]:.2f}..{cL[1]:.2f}) -> {"HOLDS" if HD else "FAILS"}')
H = {'H-H5b': HB, 'H-H5c': HC, 'H-H5d': HD}; say(f'{sum(H.values())} of 3 hold. Failed hypotheses are reported as failed; nothing was retuned. History stays context; an Alert on genuine data is a false-alarm tendency, not manipulation.')
PRODUCT = {'show_alert_followed_count': bool(HB and HD), 'k': KC, 'weighting': WC}
RES = {'protocol_h2_v1_sha256': PHH, 'k': KC, 'weighting': WC, 'hypotheses': H, 'auc': [A_, [float(x) for x in cA]], 'gain': [G_, [float(x) for x in cG]], 'lift': [L_, [float(x) for x in cL]], 'decidable': bool(DECIDABLE), 'product': PRODUCT, 'thr90_cal': THR90, 'mock': MOCK}
(OUT/'h2_results.json').write_text(json.dumps(RES, indent=1, default=str)); (OUT/'h2_share_block.txt').write_text('\n'.join(LINES))
say('Saved: h2_results.json, declared_history_alert_v1.json, h2_per_symbol.csv, h2_share_block.txt in models_b2_v1/ . Send the share block to Claude.')


CAL genuine hourly queries per symbol {'BTCUSDT': 717, 'ETHUSDT': 717}; hours followed by an Alert {'BTCUSDT': 53, 'ETHUSDT': 47}
--- score selection on CAL genuine hourly queries (TEST not evaluated yet) ---
 k weighting  CAL AUC (mean of BTC, ETH)
10         u                      0.5973
10         d                      0.6049
20         u                      0.5881
20         d                      0.5976
40         u                      0.5826
40         d                      0.5942
chosen: k = 10, weighting = d
TEST-B genuine hourly queries per symbol {'BTCUSDT': 1439, 'ETHUSDT': 1439}; hours followed by an Alert {'BTCUSDT': 121, 'ETHUSDT': 104}

--- TEST-B genuine hourly queries ---
 symbol  queries  positive hours  AUC of p_hat  AUC persistence only  gain from adding p_hat  lift top group  share flagged
BTCUSDT     1439             121         0.485                 0.529                  -0.010           0.706          0.102
ETHUSDT     1439             104         0.529    